In [1]:
# Install AutoGen Studio and pyngrok
!pip install -q autogenstudio pyngrok

# Verify the installation
!autogenstudio version


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


AutoGen Studio  CLI version: 0.4.2.2


In [5]:
import os
import getpass

# 1. Set up Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Set up Ngrok Auth Token (Required to tunnel the AutoGen Studio UI)
NGROK_TOKEN = getpass.getpass("Enter your Ngrok Auth Token: ")
!ngrok config add-authtoken {NGROK_TOKEN}

Enter your Ngrok Auth Token: ··········
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
import multiprocessing
import time
from pyngrok import ngrok

def run_autogen_studio():
    # Launch AutoGen Studio on port 8081
    !autogenstudio ui --port 8081 --host 0.0.0.0

# Start AutoGen Studio in a background process
process = multiprocessing.Process(target=run_autogen_studio)
process.start()

# Wait a moment for the server to spin up
time.sleep(5)

# Open the ngrok tunnel to port 8081
public_url = ngrok.connect(8081)
print("\n" + "="*60)
print(f"[SUCCESS] AutoGen Studio is running!")
print(f"Click the link below to open the UI:")
print(f"{public_url}")
print("="*60 + "\n")


[SUCCESS] AutoGen Studio is running!
Click the link below to open the UI:
NgrokTunnel: "https://wired-awhile-dropout.ngrok-free.dev" -> "http://localhost:8081"



In [13]:
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def run_team():
    # Fetch the Groq API key you securely entered in Cell 2
    groq_key = ""

    # Define Groq model clients with the correct model_info settings
    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # Define the individual Agents
    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."
    )

    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."
    )

    # Orchestrate the workflow team
    team = RoundRobinGroupChat(
        participants=[researcher, editor],
        termination_condition=MaxMessageTermination(max_messages=4)
    )

    # Run the prompt
    print("--- Starting Multi-Agent Session ---")
    async for message in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):
        print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
        print("-" * 40)

# Execute the async loop inside Google Colab
await run_team()

--- Starting Multi-Agent Session ---

[user]: Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs.
----------------------------------------

[Researcher]: **Groq LPUs: Enabling High-Throughput Large Language Models**

### Introduction

In recent years, Large Language Models (LLMs) have gained significant attention for their ability to process and generate human-like language. However, training and running these models require immense computational resources, leading to a demand for more efficient and specialized hardware.

**Groq LPUs: A Breakthrough in Custom Hardware Design**
---------------------------------------------------

[Groq LPUs](https://www.groq.com/) (Large Processor Arrays) are a new class of custom-built processors designed specifically for high-throughput applications such as LLMs. These LPUs are built around a scalable, massively parallel architecture, which provides a significant advantage over standard GPUs in certain workloads.

### Key Feat

AttributeError: 'TaskResult' object has no attribute 'source'